# M7.1 — Catalog loading from stored Excel files

Plan: [`plans/milestone_07/07_catalog_task_dataset_plan.md`](../../plans/milestone_07/07_catalog_task_dataset_plan.md).  
Next: `07_2_acquisition_shape_consistency.ipynb`.

Phase 1 loads the catalog from packaged `m6_matrix_plan.xlsx` (`tomography_ml_validation` test_data). Each enabled sequence → one catalog job. Singleton matrix rows have `n_particles == 1`; multi-particle groups are illustrated in `07_4` / `07_5`.

`optical_setups.source_intensity` is required (default `1.0`); it participates in clean-optical cache identity. No detector-gain column — camera roles are ray flux (prefer float `.raw.tif` / `raw_float` for absolute intensity).


In [1]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[dl,dev]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={display_path(ROOT)}")



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ROOT=.


In [2]:
from tomography_ml_validation.milestone_07 import validation_fixture_paths

paths = validation_fixture_paths()
VALIDATION_ROOT = paths["validation_root"]
WORKBOOK_PATH = paths["workbook_path"]
OUTPUT_ROOT = paths["output_root"]
CACHE_ROOT = paths["cache_root"]
print(f"workbook={display_path(WORKBOOK_PATH)}")


workbook=venv/lib/python3.12/site-packages/tomography_ml_validation/test_data/configs/m6/m6_matrix_plan.xlsx


In [3]:
from IPython.display import display

from tomography_ml.gummybear_data_catalog.gummybear_adapter import (
    catalog_jobs_to_dataframe,
    load_catalog_jobs,
)
import tomography_ml_validation.milestone_07.validation as m7_validation
from tomography_ml_validation.milestone_07 import (
    CATALOG_JOB_DISPLAY_COLUMNS,
    select_columns,
)
from gummybear_validation.notebook_tools import run_installed_pytest_test


## Job catalog reconstitution

Membership = enabled workbook jobs (`plan.jobs`), not directory scanning.


In [4]:
jobs = load_catalog_jobs(WORKBOOK_PATH, VALIDATION_ROOT)
sequence_rows = catalog_jobs_to_dataframe(jobs)
display(select_columns(sequence_rows, CATALOG_JOB_DISPLAY_COLUMNS))


,sample_id,sequence_id,split,particle_setup_id,particle_group_id,n_particles,output_root,sequence_dir,camera_schedule_id,diffusion_setup_id,selected_status,disabled_reported
0,0,bear_m6_matrix_001,train,particle_matrix_sphere_001,particle_matrix_sphere_001,1,data/generated/m6_5,/Users/thomasbraschler/git/gummybear-tomograph...,orbit_matrix_006,diff_matrix_robin_l05,workbook_enabled,False
1,1,bear_m6_matrix_002,train,particle_matrix_sphere_001,particle_matrix_sphere_001,1,data/generated/m6_5,/Users/thomasbraschler/git/gummybear-tomograph...,orbit_matrix_012,diff_matrix_robin_l05,workbook_enabled,False
2,2,bear_m6_matrix_003,train,particle_matrix_sphere_001,particle_matrix_sphere_001,1,data/generated/m6_5,/Users/thomasbraschler/git/gummybear-tomograph...,orbit_matrix_012,diff_matrix_robin_l12,workbook_enabled,False


In [5]:
assert all(int(n) == 1 for n in sequence_rows["n_particles"])
assert sequence_rows["particle_group_id"].notna().all()
print("workbook loading particle-field checks passed")


workbook loading particle-field checks passed


## Workbook-to-catalog validation


In [6]:
run_installed_pytest_test(m7_validation, "test_m7_1_catalog_membership_contract")


M7.1
Test executed: test_m7_1_catalog_membership_contract()

pytest:
../../venv/lib/python3.12/site-packages/tomography_ml_validation/milestone_07/validation.py . [100%]
============================== 1 passed in 1.78s ===============================

Test proves: Enabled GenerationPlan.jobs become catalog rows 1:1; disabled sequence IDs do not
             appear.
